<a href="https://colab.research.google.com/github/saim9211/Machine-Learning-Stuff/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [73]:
df=pd.read_csv("train.csv")
df.sample(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
149,150,0,2,"Byles, Rev. Thomas Roussel Davids",male,42.0,0,0,244310,13.0000,NaN,S
154,155,0,3,"Olsen, Mr. Ole Martin",male,NaN,0,0,Fa 265302,7.3125,NaN,S
678,679,0,3,"Goodwin, Mrs. Frederick (Augusta Tyler)",female,43.0,1,6,CA 2144,46.9000,NaN,S


In [74]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [75]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [76]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [78]:
x_train,x_test,y_train,y_test=train_test_split(df.drop("Survived", axis=1),df["Survived"],test_size=0.3,random_state=42)

In [79]:
x_test.shape,x_train.shape

((268, 7), (623, 7))

In [95]:

from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

# Define pipelines for each type of transformation

# Impute Age and then apply MinMaxScaler
age_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

# Impute Embarked, then Ordinal Encode, then MinMaxScaler
embarked_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['S', 'C', 'Q']])),
    ('minmax', MinMaxScaler())
])

# OneHotEncode Sex, then MinMaxScaler
sex_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, drop='first')),
    ('minmax', MinMaxScaler())
])

# Apply MinMaxScaler to Pclass, SibSp, Parch, and Fare
numeric_pipeline = Pipeline(steps=[
    ('scaler', MinMaxScaler()) # Now applies to Pclass, SibSp, Parch, and Fare
])

# Combine all transformations into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('age_transform', age_pipeline, ['Age']),
        ('embarked_transform', embarked_pipeline, ['Embarked']),
        ('sex_transform', sex_pipeline, ['Sex']),
        ('numeric_transform', numeric_pipeline, ['Pclass', 'SibSp', 'Parch', 'Fare']) # Included 'Fare' here
    ],
    remainder='drop'
).set_output(transform="pandas")

tf5 = SelectKBest(score_func=f_classif,k=7) # Changed score_func to f_classif, k=7
tf6 = DecisionTreeClassifier()

In [85]:
tf5

SelectKBest(k=8, score_func=<function chi2 at 0x7ec6492aa980>)

In [92]:
tf6

DecisionTreeClassifier()

In [96]:
from sklearn.pipeline import Pipeline,make_pipeline
pipe=Pipeline([
    ("preprocessor", preprocessor),
    ("tf5",tf5),
    ("tf6",tf6)
])

In [ ]:
pipe

In [97]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('age_transform',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Age']),
                                                 ('embarked_transform',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(categories=[['S',
                                                                                               'C',
                                                                                               'Q']])),
                                                                  ('minmax',
                                                                   MinMaxScaler())]),
                                                  ['Embarked']),
                                                 ('sex_transform',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False)),
                                                                  ('minmax',
                                                                   MinMaxScaler())]),
                                                  ['Sex']),
                                                 ('numeric_transform',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass', 'SibSp', 'Parch',
                                                   'Fare'])])),
                ('tf5', SelectKBest(k=7)), ('tf6', DecisionTreeClassifier())])

In [98]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = pipe.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.7537
Precision: 0.7143
Recall: 0.6757
F1-Score: 0.6944


In [99]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Define the parameter grid for the DecisionTreeClassifier (tf6)
param_grid = {
    'tf6__max_depth': [3, 5, 7, 10, None],  # Max depth of the tree
    'tf6__min_samples_leaf': [1, 5, 10, 20], # Minimum samples required at a leaf node
    'tf6__criterion': ['gini', 'entropy'] # Function to measure the quality of a split
}

# Set up StratifiedKFold for cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create GridSearchCV object
grid_search = GridSearchCV(estimator=pipe,
                           param_grid=param_grid,
                           cv=kfold,
                           scoring='accuracy',
                           n_jobs=-1,
                           verbose=1)

# Fit the GridSearchCV to the training data
grid_search.fit(x_train, y_train)

# Print the best parameters and best score
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# You can also get the best estimator
best_estimator = grid_search.best_estimator_


Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best parameters found: {'tf6__criterion': 'entropy', 'tf6__max_depth': 10, 'tf6__min_samples_leaf': 5}
Best cross-validation accuracy: 0.8281


In [102]:
import pickle

# 'grid_search' is your GridSearchCV object from the previous step
final_model = grid_search.best_estimator_

# Save (Serialize) the pipeline to a file
with open('pipeline_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

print("Pipeline saved successfully as a .pkl file!")

Pipeline saved successfully as a .pkl file!


In [103]:
import pickle
import pandas as pd

# Load the saved pipeline model
with open('pipeline_model.pkl', 'rb') as file:
    loaded_pipeline = pickle.load(file)

print("Pipeline model loaded successfully!")

Pipeline model loaded successfully!


Now that the pipeline is loaded, we can use it to make predictions on new, unseen data. For demonstration purposes, let's take a sample from our `x_test` dataset to simulate new data.